# Solutions — Diagnostic 5 — Data Processing

Exercise notebook: `00_diagnostic/diagnostic_05_data_processing.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Question 1 — 🟢 Easy · CSV totals by category

**Working solution**

In [ ]:
import csv
from collections import defaultdict

def totals_by_category(path):
    totals = defaultdict(float)
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            try:
                totals[row["category"]] += float(row["amount"])
            except (ValueError, TypeError):
                continue
    return {k: round(v, 2) for k, v in sorted(totals.items())}

**Explanation**

`csv.DictReader` + `defaultdict` + narrow `try/except`.

**Complexity:** O(n).

**Common mistakes**

- `line.split(',')`.

## Question 2 — 🟡 Medium · Clean and reject

**Working solution**

In [ ]:
def clean(records):
    good, bad = [], []
    for r in records:
        email = str(r.get("email") or "").strip().lower()
        if "@" not in email:
            bad.append({**r, "reason": "bad email"})
            continue
        try:
            age = int(r.get("age"))
        except (TypeError, ValueError):
            bad.append({**r, "reason": "bad age"})
            continue
        if not 0 <= age <= 120:
            bad.append({**r, "reason": "age out of range"})
            continue
        good.append({**r, "email": email, "country": str(r.get("country", "")).strip().upper(), "age": age})
    return good, bad

**Explanation**

Guard clauses with `continue`, new dicts via `{**r, ...}`, tuple return.

**Complexity:** O(n).

**Common mistakes**

- Mutating input records.

## Question 3 — 🟡 Medium · Latest record per key

**Working solution**

In [ ]:
def latest_by_key(records, key, ts):
    best = {}
    for r in records:
        k = r[key]
        if k not in best or r[ts] >= best[k][ts]:
            best[k] = r
    return [best[k] for k in sorted(best)]

**Explanation**

One pass with a dict of 'best so far'.

**Complexity:** O(n + k log k).

**Common mistakes**

- Keeping the first instead of the latest.

## Question 4 — 🟠 Hard · A mini ETL

**Working solution**

In [ ]:
import json
from pathlib import Path

def mini_etl(input_path, output_path):
    stats = {"lines": 0, "malformed": 0, "invalid": 0, "duplicates": 0}
    latest = {}
    with open(input_path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            stats["lines"] += 1
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                stats["malformed"] += 1
                continue
            if not isinstance(rec, dict):
                stats["malformed"] += 1
                continue
            amount = rec.get("amount")
            if isinstance(amount, bool) or not isinstance(amount, (int, float)) or amount < 0 or rec.get("status") not in ("PAID", "REFUNDED"):
                stats["invalid"] += 1
                continue
            prev = latest.get(rec["order_id"])
            if prev is not None:
                stats["duplicates"] += 1
                if prev["ts"] > rec["ts"]:
                    continue
            latest[rec["order_id"]] = rec
    by_customer = {}
    for rec in latest.values():
        if rec["status"] == "PAID":
            by_customer[rec["customer"]] = by_customer.get(rec["customer"], 0) + rec["amount"]
    summary = {**stats, "orders": len(latest), "revenue": round(sum(by_customer.values()), 2),
               "by_customer": {c: round(v, 2) for c, v in sorted(by_customer.items())}}
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    Path(output_path).write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
    return summary

**Explanation**

Read → validate → dedupe → aggregate → write, with counters for every way a line can leave the pipeline. If you finished this cleanly, Levels 6 and 8 will go fast.

**Complexity:** O(n).

**Common mistakes**

- Counting duplicates of invalid rows.
- Losing malformed lines silently.